# 🚀 Zero to Hero: Prompt Engineering for AI Developers — Part 1

> **Audience:** Python developers who want to become production-grade AI Engineers
> **Format:** Theory + hands-on, runnable code for every concept
> **Scope of Part 1:** Foundations → Setup → Zero/Few-shot → Chain-of-Thought → Role Prompting → Structured Output → Templates & Chaining → **ReAct Pattern**
> **Part 2 (separate notebook):** Meta-Prompting, Self-Critique, Evaluation/A-B testing, Production Prompt Management, Anti-Patterns, Capstone exercises

---

## Why this notebook exists

Large Language Models (LLMs) are **general-purpose reasoning engines that you program with natural language**. The quality of everything you build on top of an LLM — chatbots, autonomous agents, RAG pipelines, copilots — is bounded by the quality of the prompts and the surrounding "prompt program" (templates, parsing, validation, loops) that you write.

**Prompt Engineering is a real engineering discipline**, not "talking nicely to a chatbot". It has:
- **Theory** — a mental model of what an LLM is actually doing when it predicts text, and why certain prompt structures reliably change its behavior.
- **Patterns** — reusable techniques (few-shot, CoT, ReAct, self-consistency, etc.) backed by published research.
- **Engineering practice** — versioning, testing, evaluation, cost/latency budgets, structured outputs, and production-grade tooling.

This notebook is structured so that **every theory section is immediately followed by runnable code** that demonstrates the concept on a realistic, AI-Agent-flavored example (token costs, code review agents, multi-tool reasoning, etc.) — the kind of problems you will face when building real AI Agents.

## Mental model: an LLM is a function

```
output = f(system_prompt, conversation_history, user_input, sampling_params)
```

Prompt Engineering = **deliberately engineering every argument of `f`** so the output distribution is shifted toward what you want, as reliably and cheaply as possible.

Two consequences follow immediately:
1. **Determinism is a spectrum, not a binary.** LLMs sample from a probability distribution over tokens. Even with the same prompt, output can vary unless you control `temperature`, `top_p`, and `seed`.
2. **Everything is part of the prompt.** The system message, the few-shot examples, the conversation history, even the *order* of information — all of it is "the prompt" from the model's point of view. There is no hidden side-channel; if the model doesn't see it in the context window, it doesn't know it.

## 🗺️ Roadmap Overview

```
Phase 1 (Weeks 1–2) ← YOU ARE HERE
├── LLM APIs (OpenAI · Anthropic · Gemini)
├── Prompt Engineering ← THIS NOTEBOOK (Part 1 + Part 2)
│   ├── Zero/Few-shot · Chain-of-Thought · Role Prompting
│   └── Structured Output · Templates · ReAct · Self-Critique · Meta-Prompting
├── Tokens & Context Window Management
├── Embeddings & Semantic Search
└── Structured Output & Function/Tool Calling

Phase 2 → Tool Use · Agentic Loops · RAG · Vector DBs
Phase 3 → LangChain · LangGraph · CrewAI · Production Deployment
Phase 4 → Multi-Agent Systems · Planning · Human-in-the-loop
Phase 5 → Safety, Alignment, Fine-tuning · Frontier Research
```

## 📚 Contents of this notebook (Part 1)

| # | Topic | Techniques covered |
|---|-------|--------------------|
| 0 | What is Prompt Engineering | LLM mental model, tokens, context window |
| 1 | Setup & OpenAI API Basics | Auth, models, sampling parameters, tokenizer |
| 2 | Zero-shot / One-shot / Few-shot Prompting | In-context learning, dynamic example selection, embeddings |
| 3 | Chain-of-Thought (CoT) | Zero-shot CoT, Few-shot CoT, Self-Consistency, Tree-of-Thought |
| 4 | Role & System Prompt Design | Persona engineering, system prompt anatomy, multi-agent personas |
| 5 | Structured Output & JSON Mode | JSON schemas, Pydantic validation, intro to function/tool calling |
| 6 | Prompt Templates & Chaining | Jinja2 templating, multi-step prompt chains |
| 7 | ReAct Pattern | Reason → Act → Observe loop, building a tool-using agent from scratch |

**Part 2** will cover: Meta-Prompting, Self-Critique & Reflection, Prompt Evaluation & A/B Testing, the Production "Smart Prompt Manager", Anti-Patterns, and a capstone set of exercises.


---
## 📖 Part 0: What Is Prompt Engineering, Really?

### 0.1 How an LLM generates text

A modern LLM (GPT-4o, Claude, Gemini, ...) is an **autoregressive Transformer**: given a sequence of tokens, it predicts a probability distribution over the *next* token, samples one, appends it, and repeats.

```
P(token_n | token_1, token_2, ..., token_{n-1})
```

Everything you put in the prompt — instructions, examples, conversation history — becomes part of `token_1 ... token_{n-1}`: the **conditioning context**. The model has no other source of information. It cannot "remember" a previous session, it cannot "know" your intent unless it's written down, and it cannot use a tool unless you (the application code) detect the model's request and execute it.

This single fact explains almost every prompt engineering technique you will learn:
- **Few-shot prompting** works because the model conditions on the examples you provide — it is doing in-context pattern completion, not "learning" in the gradient-descent sense.
- **Chain-of-Thought** works because forcing the model to emit intermediate reasoning tokens gives it more "compute" (more conditioning context) before it has to commit to a final answer.
- **Role prompting** works because the pretraining + RLHF corpus contains many examples of experts behaving in certain ways; describing a role shifts the probability distribution toward that behavioral pattern.
- **ReAct** works because interleaving reasoning text with real tool outputs keeps the model's "beliefs" grounded in actual data instead of hallucinated facts.

### 0.2 Tokens & the context window

LLMs don't see characters or words — they see **tokens**, sub-word units produced by a tokenizer (e.g., `tiktoken` for OpenAI models). Understanding tokens matters for three practical reasons:

1. **Cost** — API pricing is per 1M input/output tokens.
2. **Context window limits** — every model has a maximum number of tokens it can attend to (system + history + user input + generated output). Exceed it, and you get a hard error or silently truncated content depending on the SDK.
3. **Behavior** — some prompt formats tokenize more efficiently (and sometimes more *predictably*) than others. E.g., JSON keys, special delimiters, and markdown headers tend to tokenize as stable, recognizable patterns the model has seen a lot of during training.

### 0.3 The four levers of Prompt Engineering

| Lever | What it controls | Examples in this notebook |
|-------|-------------------|----------------------------|
| **Content** | What information/instructions are present | System prompt anatomy (Part 4), structured schemas (Part 5) |
| **Structure** | How information is organized/delimited | Templates (Part 6), few-shot formatting (Part 2) |
| **Sampling** | How the model turns probabilities into tokens | `temperature`, `top_p`, `seed` (Part 1) |
| **Control flow** | How many LLM calls, in what order, with what feedback | Prompt chaining (Part 6), ReAct loop (Part 7), self-consistency (Part 3) |

Keep this table in mind: as you go through the notebook, classify every new technique into one (or more) of these four levers. It will help you reason about *why* a technique works, not just *that* it works — which is the difference between a junior and a senior prompt engineer.


---
## 🔧 Part 1: Setup & OpenAI API Basics

### 1.1 What you need

- An OpenAI API key (https://platform.openai.com/api-keys). Anthropic/Gemini equivalents follow the same concepts; we use OpenAI here because its API is the most widely taught, but **every technique in this notebook is model-agnostic**.
- Python 3.10+
- `openai`, `tiktoken`, `jinja2`, `pydantic` — installed below.

### 1.2 Security note

**Never hardcode API keys in notebooks you might commit to git.** Always prefer environment variables. We fall back to a placeholder string only so the notebook doesn't crash if you haven't set anything yet — replace it with your real key (or, better, `export OPENAI_API_KEY=...` in your shell before launching Jupyter).


In [ ]:
# Install required libraries
!pip install openai tiktoken jinja2 pydantic --quiet


In [ ]:
import os
import json
import re
import time
import asyncio
import hashlib
import statistics
from typing import Optional, Callable
from dataclasses import dataclass, field

import tiktoken
from jinja2 import Template
from pydantic import BaseModel, Field
from openai import OpenAI, AsyncOpenAI

# ============================================================
# ⚙️ CONFIGURATION: put your API key here
# ============================================================
# Option 1 (recommended): set an environment variable
#   export OPENAI_API_KEY='sk-...'
# Option 2: paste it directly below (NEVER commit this to git)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "sk-YOUR_KEY_HERE")

client = OpenAI(api_key=OPENAI_API_KEY)
async_client = AsyncOpenAI(api_key=OPENAI_API_KEY)

# Default model — switch to gpt-4o, gpt-4-turbo, gpt-3.5-turbo, etc.
DEFAULT_MODEL = "gpt-4o-mini"   # cheap, good for learning
SMART_MODEL   = "gpt-4o"        # stronger, used for harder reasoning tasks

print("✅ OpenAI client initialized!")
print(f"   Default model : {DEFAULT_MODEL}")
print(f"   Smart model   : {SMART_MODEL}")


### 1.3 The most important sampling parameters

| Parameter | Meaning | Recommended value |
|-----------|---------|--------------------|
| `temperature` | Randomness of sampling (0 = near-deterministic, 2 = very random) | 0.0–0.2 for production tasks that need consistency; 0.7–1.0 for creative tasks |
| `top_p` | Nucleus sampling — only sample from the smallest set of tokens whose cumulative probability ≥ `top_p` | Use **either** `temperature` **or** `top_p`, rarely both at non-default values |
| `max_tokens` | Hard cap on the length of the generated output | Always set explicitly — never rely on the default |
| `frequency_penalty` | Penalizes tokens proportionally to how often they've already appeared (0–2) | 0.3–0.5 for long creative text to reduce repetition |
| `presence_penalty` | Penalizes tokens that have appeared at all, encouraging new topics (0–2) | 0.1–0.3 |
| `seed` | Best-effort determinism — same seed + same params ⇒ (usually) same output | Any integer, useful for reproducible tests |
| `response_format` | Forces valid JSON output | `{"type": "json_object"}` (see Part 5) |

> ⚠️ **Common misconception:** `temperature=0` does **not** guarantee bit-for-bit identical output across calls (GPU non-determinism, load balancing, etc.), but it gets you very close, and is the right default for anything where consistency matters (classification, extraction, code review, etc.).


In [ ]:
# ============================================================
# 🛠️ Helper function used throughout this notebook
# ============================================================

def chat(
    user_message: str,
    system_message: str = "You are a helpful AI assistant.",
    model: str = DEFAULT_MODEL,
    temperature: float = 0.1,
    max_tokens: int = 1000,
    verbose: bool = True
) -> str:
    """
    Thin wrapper around the OpenAI Chat Completions API.
    Returns the text response and (optionally) prints token usage.
    """
    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user",   "content": user_message}
        ]
    )
    result = response.choices[0].message.content
    if verbose:
        usage = response.usage
        print(f"📊 Tokens — Input: {usage.prompt_tokens} | Output: {usage.completion_tokens} | Total: {usage.total_tokens}")
        print("-" * 60)
        print(result)
    return result


def count_tokens(text: str, model: str = "gpt-4o") -> int:
    """Count the number of tokens in a piece of text."""
    enc = tiktoken.encoding_for_model(model)
    return len(enc.encode(text))


print("✅ Helper functions are ready!")

# Quick sanity check
test_text = "Hello, I am learning AI Agents!"
print(f"Token count for '{test_text}': {count_tokens(test_text)} tokens")


### 1.4 Tokenizer deep-dive (hands-on)

Let's actually look *inside* the tokenizer to build intuition: how text is chunked, why some languages/strings cost more tokens than others, and why this matters for cost and context-window budgeting.


In [ ]:
# ============================================================
# 🧪 PRACTICE: Looking inside the tokenizer
# ============================================================
enc = tiktoken.encoding_for_model("gpt-4o")

samples = [
    "Hello world",
    "AI Agents",
    "prompt engineering",
    "Xin chào, tôi đang học AI Agents!",   # non-English text usually costs MORE tokens
    "function_call(param1='value', param2=42)",  # code-like text tokenizes differently than prose
]

for s in samples:
    ids = enc.encode(s)
    decoded_pieces = [enc.decode([t]) for t in ids]
    print(f"Text: {s!r}")
    print(f"  → {len(ids)} tokens: {decoded_pieces}")
    print()

# 🎯 Exercise: Try a long English sentence vs. the same sentence in Vietnamese.
# Compare token counts — this directly affects API cost when you serve multilingual users.


In [ ]:
# 🧪 PRACTICE: Comparing temperature
prompt = "Write one sentence describing what an AI Agent is."

print("=" * 60)
print("Temperature = 0.0 (near-deterministic):")
_ = chat(prompt, temperature=0.0)

print("\n" + "=" * 60)
print("Temperature = 1.0 (creative / diverse):")
_ = chat(prompt, temperature=1.0)

# 🎯 Exercise: Run this cell several times.
# temperature=0.0 → output should stay (almost) identical every run
# temperature=1.0 → output should vary noticeably every run


In [ ]:
# 🧪 PRACTICE: seed for reproducibility, and top_p as an alternative to temperature
print("Run A (seed=42):")
resp_a = client.chat.completions.create(
    model=DEFAULT_MODEL, temperature=0.8, seed=42, max_tokens=60,
    messages=[{"role": "user", "content": prompt}]
)
print(resp_a.choices[0].message.content)

print("\nRun B (same seed=42):")
resp_b = client.chat.completions.create(
    model=DEFAULT_MODEL, temperature=0.8, seed=42, max_tokens=60,
    messages=[{"role": "user", "content": prompt}]
)
print(resp_b.choices[0].message.content)

print("\n💡 With the same seed + same parameters, outputs should be identical or very close.")
print("   This is essential for writing reproducible tests for your prompts (see Part 2: Evaluation).")

# 🎯 Exercise: Replace temperature=0.8 with top_p=0.9 (and remove temperature) and observe the difference.


---
## 🎯 Part 2: Zero-shot, One-shot & Few-shot Prompting

### 2.1 Theory: In-Context Learning

LLMs exhibit **in-context learning (ICL)**: the ability to perform a new task by conditioning on a handful of input→output examples *inside the prompt*, with **no weight updates**. This is fundamentally different from traditional ML "training":

| | Traditional ML training | In-context learning |
|---|---|---|
| Mechanism | Gradient descent updates model weights | Model attends to examples in the context window |
| Persistence | Permanent (until retrained) | Only for the duration of that single prompt |
| Data needed | Hundreds to millions of examples | Often 0–10 examples |
| Cost | Expensive (GPU-hours) | Just the token cost of the extra examples |

Three flavors:
- **Zero-shot** — no examples, just an instruction. Relies entirely on the model's pretraining "knowledge" of the task pattern.
- **One-shot** — exactly one example. Useful for showing the *format* you want without spending too many tokens.
- **Few-shot** — multiple (typically 3–10) examples. The most powerful and the most commonly used in production for tasks needing a precise output format or a non-obvious classification boundary.

### 2.2 When zero-shot fails and few-shot saves you

Zero-shot is the **right default** to try first — it's cheapest. Move to few-shot when you observe:
- Inconsistent output **format** (the model sometimes adds extra commentary, sometimes doesn't)
- Domain-specific or ambiguous **labels** that aren't obvious from the label name alone
- A **house style** you need the model to imitate exactly


In [ ]:
# ============================================================
# 2.1 ZERO-SHOT PROMPTING
# No examples — the model relies entirely on pretraining knowledge
# ============================================================

zero_shot_prompt = """
Classify the sentiment of the following review:

Review: "The product is bad, not worth the money, slow delivery"
Sentiment (POSITIVE/NEGATIVE/NEUTRAL):
"""

print("🔍 ZERO-SHOT:")
_ = chat(zero_shot_prompt)


In [ ]:
# ============================================================
# 2.1.b ONE-SHOT PROMPTING
# Exactly one example — mainly used to pin down the OUTPUT FORMAT
# ============================================================

one_shot_prompt = """
Classify the sentiment and output EXACTLY in this format.

Example:
Review: "Fast shipping, great packaging, exactly as described"
Sentiment: POSITIVE | Confidence: 0.95

Now classify:
Review: "The product is bad, not worth the money, slow delivery"
Sentiment:
"""

print("🔍 ONE-SHOT:")
_ = chat(one_shot_prompt)

# 💡 Notice how much more reliable the output FORMAT is compared to zero-shot,
# even though we only added a single example.


In [ ]:
# ============================================================
# 2.2 FEW-SHOT PROMPTING — Anatomy of a Good Few-shot Prompt
# Provide examples to teach the model the desired format and decision pattern
# ============================================================

system_prompt = "You are a sentiment analysis expert for an e-commerce platform."

few_shot_prompt = """
# TASK: Analyze the sentiment of a product review
# FORMAT: Label (POSITIVE/NEGATIVE/NEUTRAL) + Score (0.0-1.0) + Reason (1 sentence)

---
# Example 1:
Review: "Fast delivery, carefully packaged, product exactly as described"
Label: POSITIVE
Score: 0.95
Reason: Mentions 3 important positive factors: speed, packaging, and quality

# Example 2:
Review: "Quality is okay but the price is a bit high compared to the market"
Label: NEUTRAL
Score: 0.45
Reason: Has a positive point (quality) but also a negative one (price)

# Example 3:
Review: "Poor quality product, shop doesn't support customers, very disappointed"
Label: NEGATIVE
Score: 0.05
Reason: Multiple serious issues: poor quality and bad service

---
# Now analyze:
Review: "Nice design but battery doesn't match the advertisement, a bit disappointed"
Label:
Score:
Reason:
"""

print("🔍 FEW-SHOT (3 examples):")
_ = chat(few_shot_prompt, system_message=system_prompt)


In [ ]:
# ============================================================
# 2.3 DYNAMIC FEW-SHOT SELECTION
# Advanced technique: pick the examples most relevant to the current input
# In production, this is done with embedding similarity (we show both versions below)
# ============================================================

EXAMPLE_BANK = [
    {"review": "Great battery life, nice camera, sharp display, totally worth buying",
     "category": "phone", "label": "POSITIVE", "score": 0.92},
    {"review": "Laptop has poor cooling, gets hot after long use",
     "category": "laptop", "label": "NEGATIVE", "score": 0.20},
    {"review": "Good sound quality, stable connection, battery lasts about 8 hours",
     "category": "earphone", "label": "POSITIVE", "score": 0.80},
    {"review": "Phone is okay but screen lags during heavy gaming",
     "category": "phone", "label": "NEUTRAL", "score": 0.50},
]


def select_examples_keyword(query: str, bank: list, n: int = 2) -> list:
    """
    Naive keyword-overlap selection (production systems use embedding cosine similarity).
    Picks the n examples most related to the query.
    """
    keywords = set(query.lower().split())
    scored = []
    for ex in bank:
        ex_words = set(ex["review"].lower().split())
        overlap = len(keywords & ex_words)
        scored.append((overlap, ex))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [ex for _, ex in scored[:n]]


def build_dynamic_few_shot_prompt(query: str, n_examples: int = 2) -> str:
    examples = select_examples_keyword(query, EXAMPLE_BANK, n_examples)
    parts = ["# Analyze the sentiment of an electronics product review\n"]
    for i, ex in enumerate(examples, 1):
        parts.append(f"# Example {i}:")
        parts.append(f'Review: "{ex["review"]}"')
        parts.append(f"Label: {ex['label']}")
        parts.append(f"Score: {ex['score']}\n")
    parts.append("# Now analyze:")
    parts.append(f'Review: "{query}"')
    parts.append("Label:\nScore:")
    return "\n".join(parts)


# Test dynamic selection
test_review = "Nice phone but weak battery, only lasts half a day"
dynamic_prompt = build_dynamic_few_shot_prompt(test_review)

print("📝 Generated dynamic few-shot prompt:")
print(dynamic_prompt)
print("\n" + "=" * 60)
print("🤖 Result:")
_ = chat(dynamic_prompt)


In [ ]:
# ============================================================
# 2.3.b PRODUCTION VERSION — Selection via real embeddings
# This is what you should actually use once your example bank grows beyond
# a handful of hand-picked items (keyword overlap doesn't scale or generalize).
# ============================================================
import numpy as np


def get_embedding(text: str, model: str = "text-embedding-3-small") -> list[float]:
    resp = client.embeddings.create(model=model, input=text)
    return resp.data[0].embedding


def cosine_similarity(a: list[float], b: list[float]) -> float:
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


def select_examples_embedding(query: str, bank: list, n: int = 2) -> list:
    """Select the n most semantically similar examples using embeddings."""
    query_emb = get_embedding(query)
    scored = []
    for ex in bank:
        ex_emb = get_embedding(ex["review"])
        sim = cosine_similarity(query_emb, ex_emb)
        scored.append((sim, ex))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [ex for _, ex in scored[:n]]


# Test — this call hits the embeddings API, comment out if you want to save cost
selected = select_examples_embedding(test_review, EXAMPLE_BANK, n=2)
print("📌 Examples selected by semantic similarity (embeddings):")
for ex in selected:
    print(f"  - {ex['review']}  → {ex['label']}")

# 💡 In production with thousands of examples, you would pre-compute embeddings once,
# store them in a vector DB (FAISS, Pinecone, pgvector, Qdrant...), and do a fast
# nearest-neighbor lookup at request time instead of re-embedding the whole bank every call.


### ✅ Best Practices for Few-shot Prompting

| Principle | Detail |
|-----------|--------|
| **Quality > Quantity** | 3–5 excellent examples beat 10 mediocre ones |
| **Diversity** | Cover important edge cases and variations |
| **Consistent formatting** | All examples must share the exact same structure |
| **Recency bias** | The *last* example has the strongest influence on the model's output |
| **Task before examples** | Describe the task before showing examples, not after |
| **Dynamic selection** | Selecting the most relevant examples per-input can improve accuracy by 15–20% over a fixed example set |
| **Order matters** | If you observe instability, try shuffling/reordering examples — model sensitivity to order is a known phenomenon (see Lu et al., 2021, "Fantastically Ordered Prompts") |


In [ ]:
# 🎯 EXERCISE 2: Build your own few-shot prompt
# Task: classify a customer support message into one of:
#   BILLING / TECHNICAL / SHIPPING / GENERAL
# Write at least 3 examples below, then test on the sample message.

your_few_shot_prompt = """
# TODO: write your few-shot prompt here, following the anatomy shown above
"""

sample_message = "My package was supposed to arrive 5 days ago and tracking hasn't updated since."

# Uncomment once your prompt is ready:
# _ = chat(your_few_shot_prompt + f'\nMessage: "{sample_message}"\nCategory:')


---
## 🧠 Part 3: Chain-of-Thought (CoT) Prompting

### 3.1 Theory: why "thinking step by step" actually helps

Wei et al. (2022), *"Chain-of-Thought Prompting Elicits Reasoning in Large Language Models"*, showed that simply asking a model to produce intermediate reasoning steps before its final answer dramatically improved accuracy on arithmetic, commonsense, and symbolic reasoning tasks (e.g., from ~18% to ~57% on certain math benchmarks for the models tested at the time).

Why does this work, mechanistically? Two complementary explanations:

1. **More "compute" per answer.** An autoregressive model can only do a fixed amount of "thinking" per generated token (one forward pass). If you force it to answer immediately, it has very little budget. If you let it write out reasoning tokens first, each of those tokens becomes new context that informs the next one — effectively giving the model many more forward passes to work the problem out before committing to an answer.
2. **Distributional shift toward "showing work."** Pretraining data contains huge amounts of text where reasoning is written out (math solutions, tutorials, code with comments). Asking for step-by-step reasoning shifts the model into that part of its learned distribution, away from terse pattern-matching responses that are more error-prone for multi-step problems.

### 3.2 Variants of CoT

| Variant | Mechanism |
|---------|-----------|
| **Zero-shot CoT** | Append a trigger phrase like "Let's think step by step" — no examples needed |
| **Few-shot CoT** | Show worked examples *with* their reasoning chains, so the model imitates the reasoning *style*, not just the final answer |
| **Self-Consistency** | Sample the same CoT prompt N times at temperature > 0, take a majority vote over final answers |
| **Tree-of-Thought (ToT)** | Generate and evaluate multiple *branches* of reasoning, not just one linear chain, then select/backtrack |


In [ ]:
# ============================================================
# 3.1 ZERO-SHOT CoT — "Let's think step by step"
# Wei et al., 2022: boosted accuracy from ~18% to ~57% on arithmetic tasks
# ============================================================

# A realistic token-cost problem — very relevant in AI Agent development!
problem = """
An AI Agent needs to process 1,000 requests per day.
Each request consumes an average of 800 tokens (input + output combined).
GPT-4o-mini pricing: $0.00015/1K input tokens, $0.0006/1K output tokens.
Assume an input:output ratio of 70:30.
What is the total monthly cost (30 days) in USD?
"""

# Without CoT
print("❌ WITHOUT CoT:")
_ = chat(problem)

print("\n" + "=" * 60)
print("✅ WITH Zero-shot CoT:")
_ = chat(problem + "\n\nLet's think step by step before giving the final answer.")

# 🎯 Exercise: Manually verify the math. Does the CoT version get it right
# more reliably than the non-CoT version? Run both several times at temperature=0.7
# to see how often each is correct.


In [ ]:
# ============================================================
# 3.2 FEW-SHOT CoT — Teaching the reasoning PATTERN via worked examples
# ============================================================

cot_few_shot = """
Solve the following problems about AI Agent performance:

---
Q: If an agent calls 3 tools in parallel, each tool taking 2 seconds,
   but one tool times out after 5 seconds, what is the total time?
A: Let's think step by step:
   1. 3 tools run in parallel → total time = max(time of each tool)
   2. Tool 1: 2s, Tool 2: 2s, Tool 3: timeout = 5s
   3. max(2, 2, 5) = 5 seconds
   4. Add retry-logic overhead: +2s
   → Result: ~7s with retry, or 5s if fail-fast

---
Q: An agent has a 128K-token context window. Each conversation turn averages
   500 tokens. The agent must keep a 2000-token system prompt and 1000-token
   tool definitions resident at all times. What's the max number of turns?
A: Let's think step by step:
   1. Context window: 128,000 tokens
   2. Fixed overhead: system_prompt(2000) + tools(1000) = 3,000 tokens
   3. Remaining budget for conversation: 128,000 - 3,000 = 125,000 tokens
   4. Each turn: 500 tokens
   5. Max turns: 125,000 / 500 = 250 turns
   → Result: 250 turns (in practice, cap at ~200 to leave a safety buffer)

---
Q: An agent's RAG pipeline embeds 1,000 docs, each 512 tokens.
   The ada-002 embedding model costs $0.0001/1K tokens.
   What is the cost of indexing the entire corpus?
A: Let's think step by step:
"""

print("🔍 FEW-SHOT CoT:")
_ = chat(cot_few_shot, temperature=0.1)


In [ ]:
# ============================================================
# 3.3 SELF-CONSISTENCY — Majority voting to boost accuracy
# Run the same prompt N times with temperature > 0, take the most common answer
# ============================================================

from collections import Counter


def self_consistency(
    prompt: str,
    n_samples: int = 5,
    temperature: float = 0.7,
    extract_final_answer: Callable = None
) -> dict:
    """
    Run a prompt n times and return the majority answer.
    Best suited for problems with one clearly correct answer.
    Cost: n × the price of a single call — use when accuracy really matters.
    """
    answers = []
    full_responses = []

    print(f"🔄 Running {n_samples} samples at temperature={temperature}...")

    for i in range(n_samples):
        response = client.chat.completions.create(
            model=DEFAULT_MODEL,
            temperature=temperature,
            max_tokens=500,
            messages=[{"role": "user", "content": prompt}]
        )
        full_text = response.choices[0].message.content
        full_responses.append(full_text)

        if extract_final_answer:
            answer = extract_final_answer(full_text)
        else:
            answer = full_text.strip().split("\n")[-1].strip()

        answers.append(answer)
        print(f"  Sample {i+1}: {answer[:80]}..." if len(answer) > 80 else f"  Sample {i+1}: {answer}")

    counter = Counter(answers)
    majority_answer, count = counter.most_common(1)[0]

    return {
        "majority_answer": majority_answer,
        "confidence": count / n_samples,
        "all_answers": dict(counter),
        "all_responses": full_responses
    }


# Test Self-Consistency
sc_prompt = """
An AI startup has 3 engineers. Each engineer builds 2 AI agents.
Each agent makes an average of 5 API calls per request.
If the system receives 100 requests/day, what is the total number of API calls in 1 week?

Think step by step and give the final result as a single number.
"""

result = self_consistency(sc_prompt, n_samples=5)
print(f"\n📊 SELF-CONSISTENCY RESULT:")
print(f"   Majority answer : {result['majority_answer']}")
print(f"   Confidence      : {result['confidence']:.0%}")
print(f"   All answers     : {result['all_answers']}")

# 💡 Note the trade-off: confidence here is a measure of *agreement*, not
# necessarily *correctness*. Always sanity-check the majority answer yourself
# when first setting this up for a new task.


In [ ]:
# ============================================================
# 3.4 TREE-OF-THOUGHT (ToT)
# Explore multiple reasoning branches in parallel, evaluate, and pick the best
# (Yao et al., 2023, "Tree of Thoughts: Deliberate Problem Solving with LLMs")
# ============================================================

tot_prompt = """
Problem: Design a memory architecture for an AI Agent that must handle 10,000 sessions/day,
each session having 50 conversation turns.

Propose 3 DIFFERENT approaches:

Approach 1: [architecture name]
→ Description: ...
→ Pros: ...
→ Cons: ...
→ Score: .../10

Approach 2: [architecture name]
→ Description: ...
→ Pros: ...
→ Cons: ...
→ Score: .../10

Approach 3: [architecture name]
→ Description: ...
→ Pros: ...
→ Cons: ...
→ Score: .../10

Final Recommendation: [chosen approach and why]
"""

print("🌳 TREE-OF-THOUGHT:")
_ = chat(tot_prompt, model=SMART_MODEL, max_tokens=1500)

# 💡 This is a simplified, single-prompt approximation of ToT (sometimes called
# "ToT-lite"). A full ToT implementation would make SEPARATE LLM calls to
# (1) generate candidate branches, (2) score each branch independently, and
# (3) expand the most promising branch further — a literal breadth/depth-first
# search over reasoning paths, which is naturally a great fit for the
# prompt-chaining pattern you'll see in Part 6.


### 🎯 When should you use which CoT technique?

| Technique | Use when | Trade-off |
|-----------|----------|-----------|
| **Zero-shot CoT** | Simple reasoning task with one clear answer | Fast, cheap |
| **Few-shot CoT** | Domain-specific reasoning where you need to teach a specific reasoning pattern | More tokens per call |
| **Self-Consistency** | High accuracy required, latency & cost × n is acceptable | n× the cost of one call |
| **Tree-of-Thought** | Searching a large solution space for the best option | Most complex and most expensive |


In [ ]:
# 🎯 EXERCISE 3: Apply CoT to a new problem
# Task: an agent's vector DB query costs $0.02 per 1000 queries, and the agent
# makes an average of 4 queries per user request. If you have 50,000 daily
# active users each sending 3 requests/day, what is the monthly vector DB cost?
# Write a zero-shot CoT prompt AND a self-consistency check (n=3) below.

your_cot_prompt = """
# TODO: write your zero-shot CoT prompt here
"""

# Uncomment once ready:
# _ = chat(your_cot_prompt)
# result = self_consistency(your_cot_prompt, n_samples=3)
# print(result["majority_answer"])


---
## 👤 Part 4: Role Prompting & System Prompt Design

### 4.1 Theory: why role prompting works

Assigning a role ("You are a senior security engineer...") is not magic — it's a **conditioning trick** that shifts the model's output distribution toward text patterns associated with that role in its training data. A vague role like "helpful assistant" carries almost no useful conditioning signal; a precise role with explicit priorities, capabilities, and constraints gives the model a much narrower (and therefore more *predictable* and *higher-quality*) target distribution to sample from.

A good system prompt typically encodes:
- **Identity** — who the model is "pretending" to be, and its relevant expertise.
- **Objective** — what task, for what audience.
- **Capabilities** — what kinds of things it should look for / be able to do.
- **Constraints** — explicit boundaries: what NOT to do, output limits, etc.
- **Output format** — the exact structure expected, so downstream code can parse it.
- **Behavior rules** — how to resolve ambiguity or priority conflicts (e.g., "security > correctness > performance > style").

### 4.2 System prompt vs. user prompt

In multi-turn chat APIs, the `system` message is conceptually different from `user` messages: it's meant to be a stable, persistent instruction layer that frames *how* the model should behave for the entire conversation, while `user` messages carry the task-specific content. In practice, *some* models weight system messages more strongly than user messages (and some don't differentiate as much) — always test empirically with the model you're shipping with.


In [ ]:
# ============================================================
# 4.1 Comparing: Vague Role vs. Specific Role
# ============================================================

code_to_review = """
async def get_user(user_id: int, db):
    user = db.query("SELECT * FROM users WHERE id = " + str(user_id)).first()
    return user
"""

# ❌ Vague role
bad_system = "You are a helpful AI assistant."

# ✅ Specific role with context, style, and priorities
good_system = """
You are a Senior Python Engineer with 10+ years building production backend systems at scale.
You specialize in security vulnerabilities and database performance.

When reviewing code, you:
1. ALWAYS identify security vulnerabilities first (SQL injection, XSS, auth issues)
2. Then flag performance bottlenecks with specific metrics
3. Provide corrected code, not just descriptions
4. Explain WHY each issue matters in production
5. Rate severity: CRITICAL / HIGH / MEDIUM / LOW
"""

user_msg = f"Review this code:\n```python\n{code_to_review}\n```"

print("❌ BAD (vague role):")
_ = chat(user_msg, system_message=bad_system, max_tokens=300)

print("\n" + "=" * 60)
print("✅ GOOD (specific role):")
_ = chat(user_msg, system_message=good_system, max_tokens=500)

# 💡 Notice: the vague-role response is usually generic and may miss the
# SQL injection entirely. The specific-role response should call it out
# explicitly, with severity and a concrete fix (parameterized query).


In [ ]:
# ============================================================
# 4.2 ANATOMY OF A PROFESSIONAL SYSTEM PROMPT
# A full template for a production AI Agent
# ============================================================

CODE_REVIEW_AGENT_SYSTEM_PROMPT = """
# IDENTITY
You are CodeReview Agent, an expert Python code reviewer with 15 years of experience.
You have reviewed 50,000+ pull requests at FAANG-scale companies.

# OBJECTIVE
Task: Analyze provided code and deliver actionable, specific feedback.
Audience: Experienced Python developers — skip basic explanations.

# CAPABILITIES
You can identify:
- Security vulnerabilities (SQL injection, XSS, SSRF, auth bypass, etc.)
- Performance bottlenecks (N+1 queries, missing indexes, memory leaks)
- Code quality issues (naming, coupling, testability)
- Production readiness gaps (error handling, logging, monitoring)

# CONSTRAINTS
- Do NOT auto-fix code unless explicitly asked
- Do NOT comment on style unless it affects readability significantly
- Maximum 8 issues per review (prioritize by severity)
- If no issues found, explicitly state: "Approved ✅"

# OUTPUT FORMAT
Use exactly this structure:

## 📋 Summary
[1-2 sentence overview]

## 🚨 Critical Issues (must fix before merge)
- [ISSUE_TYPE] Description → Impact → Fix suggestion

## ⚠️ Suggestions (should fix)
- Description → Why it matters

## ✅ Positives
- What was done well

# BEHAVIOR RULES
- Missing tests → Always list under Critical Issues
- Security issue → ALWAYS critical, never downgrade
- Ambiguous code → Ask for clarification, don't assume intent
- Priority: security > correctness > performance > style
"""

# Test with code that has multiple issues
vulnerable_code = """
import sqlite3
import pickle

def get_user_data(username, password):
    conn = sqlite3.connect('app.db')
    cursor = conn.cursor()
    query = f"SELECT * FROM users WHERE username='{username}' AND password='{password}'"
    cursor.execute(query)
    result = cursor.fetchall()
    conn.close()
    return result

def load_user_preferences(user_id):
    with open(f'/tmp/prefs_{user_id}.pkl', 'rb') as f:
        return pickle.load(f)  # Load user preferences
"""

print("🔍 CODE REVIEW AGENT:")
_ = chat(
    f"Review this code:\n```python\n{vulnerable_code}\n```",
    system_message=CODE_REVIEW_AGENT_SYSTEM_PROMPT,
    model=SMART_MODEL,
    max_tokens=1000
)


In [ ]:
# ============================================================
# 4.3 MULTI-PERSONA prompting for Multi-Agent Systems
# Each agent needs its own clearly separated persona to avoid role confusion
# ============================================================

ORCHESTRATOR_PROMPT = """
You are the Orchestrator Agent.

Your ONLY responsibilities:
1. Receive the user's task
2. Decompose it into clear subtasks
3. Specify which specialist agent handles each subtask
4. Synthesize results into a final response

You do NOT execute subtasks yourself. You ONLY coordinate.

Output format:
{
  "task_analysis": "...",
  "subtasks": [
    {"id": 1, "agent": "researcher|coder|reviewer", "task": "...", "depends_on": []}
  ],
  "execution_order": [1, 2, 3]
}
"""

RESEARCHER_PROMPT = """
You are the Research Agent.
ONLY search for and return information when requested by the Orchestrator.
Return raw information — do NOT interpret, recommend, or add opinions.
Format: JSON with fields: source, content, confidence_score (0-1)
"""

CODER_PROMPT = """
You are the Code Generation Agent.
ONLY write code when given a clear specification.
Always include: type hints, docstring, error handling, and at least 1 usage example.
Return only the code block — no explanations unless asked.
"""

# Simulate the Orchestrator decomposing a task
user_task = """
Build a function that fetches real-time stock prices from an API
and stores them in a SQLite database with proper error handling.
"""

print("🎭 ORCHESTRATOR AGENT:")
orchestrator_result = chat(
    f"Task from user: {user_task}",
    system_message=ORCHESTRATOR_PROMPT,
    model=SMART_MODEL,
    max_tokens=600
)

# 💡 Why separate personas instead of one giant prompt?
# - Each agent's context stays small and focused → fewer hallucinations
# - You can swap models per-agent (cheap model for research, smart model for coding)
# - Easier to test, version, and debug each agent independently
# - Mirrors the design of real frameworks like AutoGen / CrewAI / LangGraph


In [ ]:
# 🎯 EXERCISE 4: Write a "Vague" and a "Specific" system prompt for a
# customer-support email-drafting agent, then compare outputs side by side
# on the same incoming complaint email below.

complaint_email = """
Subject: Order #4521 never arrived
I ordered a laptop 2 weeks ago and it still hasn't shipped. This is unacceptable,
I need a refund or my laptop shipped TODAY.
"""

vague_support_system = "You are a helpful customer support assistant."

specific_support_system = """
# TODO: design a precise system prompt:
# - Identity (e.g., senior support agent, tone, empathy level)
# - Objective (defuse frustration, offer concrete next steps)
# - Constraints (no over-promising, must offer 1-2 concrete options)
# - Output format (subject line + body)
"""

# Uncomment once ready:
# _ = chat(complaint_email, system_message=vague_support_system)
# _ = chat(complaint_email, system_message=specific_support_system)


---
## 📊 Part 5: Structured Output & JSON Mode

### 5.1 Theory: why structured output is non-negotiable for AI Agents

A human reading free-text output can tolerate ambiguity. **Your code cannot.** Every time an LLM's output feeds into another function, API call, or database write, you need a **guaranteed, parseable structure** — otherwise your pipeline breaks on edge cases like extra commentary, markdown fences around JSON, trailing commas, or inconsistent field names.

Three increasingly strict layers of structured output:

1. **Prompt-level formatting instructions** — weakest; the model *usually* complies but can drift.
2. **JSON mode** (`response_format={"type": "json_object"}`) — the API guarantees syntactically valid JSON, but **not** that it matches *your* schema.
3. **Schema validation with Pydantic** — you parse the JSON and validate it against a strict schema (types, ranges, enums, required fields). This is the **production-grade pattern**: trust nothing from the model until it has passed validation, and have a clear error-handling path (retry with the validation error fed back to the model) for when it fails.

> 🔭 **Looking ahead:** Modern APIs increasingly support native **function/tool calling** and even **strict structured outputs tied directly to a JSON schema** (e.g., OpenAI's `strict` mode, Anthropic's tool use). These push schema enforcement into the API itself rather than relying purely on prompting + post-hoc validation. We introduce the *concept* of tool calling here because the underlying mental model (the model emits a structured "intent", your code executes it, you feed back the result) is exactly what powers the ReAct pattern in Part 7 — and we revisit tool-calling APIs in more depth in Part 2.


In [ ]:
# ============================================================
# 5.1 OpenAI's JSON MODE
# response_format={"type": "json_object"} guarantees syntactically valid JSON
# ============================================================

def get_json_output(prompt: str, schema_description: str) -> dict:
    """
    Always returns valid JSON.
    Important: you MUST mention "JSON" in the system or user message when using json_object mode,
    or the API will raise an error.
    """
    response = client.chat.completions.create(
        model=DEFAULT_MODEL,
        temperature=0.1,
        max_tokens=1000,
        response_format={"type": "json_object"},  # ← key parameter!
        messages=[
            {
                "role": "system",
                "content": f"""
You are a data extraction agent.
ALWAYS respond with valid JSON following this schema:
{schema_description}
Rules:
- Output JSON only, no additional text
- Use null for missing data, never omit fields
"""
            },
            {"role": "user", "content": prompt}
        ]
    )
    return json.loads(response.choices[0].message.content)


# Test: extract structured info from unstructured text
job_posting = """
We are hiring a Senior AI Engineer with 5+ years of Python experience.
Requirements: FastAPI, LangChain, Docker, experience deploying ML models.
Salary: $3000-5000/month. Remote-first. Application deadline: 2024-12-31.
Email: career@aicompany.com
"""

schema = """
{
  "position": "string",
  "seniority": "junior|mid|senior|lead",
  "years_experience": "number or null",
  "required_skills": ["string"],
  "salary_range": {"min": "number or null", "max": "number or null", "currency": "string"},
  "work_mode": "remote|hybrid|onsite",
  "deadline": "YYYY-MM-DD or null",
  "contact": "string or null"
}
"""

result = get_json_output(f"Extract info from: {job_posting}", schema)
print("📊 Structured Output:")
print(json.dumps(result, indent=2, ensure_ascii=False))


In [ ]:
# ============================================================
# 5.2 PYDANTIC VALIDATION for structured outputs
# This is the most production-grade pattern
# ============================================================

from pydantic import BaseModel, Field, field_validator
from typing import List, Optional
from enum import Enum


class Severity(str, Enum):
    CRITICAL = "CRITICAL"
    HIGH = "HIGH"
    MEDIUM = "MEDIUM"
    LOW = "LOW"


class CodeIssue(BaseModel):
    type: str = Field(description="Issue type: SECURITY, PERFORMANCE, CORRECTNESS, STYLE")
    severity: Severity
    description: str
    line_number: Optional[int] = None
    fix_suggestion: str


class CodeReviewResult(BaseModel):
    summary: str
    approved: bool
    overall_score: int = Field(ge=0, le=10, description="0-10 score")
    issues: List[CodeIssue]
    positives: List[str]

    @field_validator("overall_score")
    @classmethod
    def validate_score(cls, v):
        if not 0 <= v <= 10:
            raise ValueError("Score must be 0-10")
        return v


def review_code_structured(code: str) -> CodeReviewResult:
    schema_json = CodeReviewResult.model_json_schema()

    response = client.chat.completions.create(
        model=SMART_MODEL,
        temperature=0.1,
        max_tokens=1500,
        response_format={"type": "json_object"},
        messages=[
            {
                "role": "system",
                "content": f"""You are an expert code reviewer.
Return JSON matching this schema exactly:
{json.dumps(schema_json, indent=2)}"""
            },
            {"role": "user", "content": f"Review this Python code:\n```python\n{code}\n```"}
        ]
    )

    raw = json.loads(response.choices[0].message.content)
    return CodeReviewResult(**raw)  # Pydantic validation happens here!


# Test
sample_code = """
def divide(a, b):
    return a / b  # No error handling

result = divide(10, 0)  # Will crash!
"""

review = review_code_structured(sample_code)
print(f"✅ Pydantic-validated output:")
print(f"   Approved : {review.approved}")
print(f"   Score    : {review.overall_score}/10")
print(f"   Issues   : {len(review.issues)}")
print(f"   Summary  : {review.summary}")
if review.issues:
    print(f"\n   Issues found:")
    for issue in review.issues:
        print(f"   [{issue.severity}] {issue.type}: {issue.description}")
        print(f"   → Fix: {issue.fix_suggestion}")


In [ ]:
# ============================================================
# 5.3 RETRY-ON-VALIDATION-FAILURE pattern
# Production systems must handle the case where the model's JSON doesn't
# match your Pydantic schema. The right move is usually: feed the validation
# error BACK to the model and ask it to fix its own output.
# ============================================================
from pydantic import ValidationError


def review_code_with_retry(code: str, max_retries: int = 2) -> CodeReviewResult:
    schema_json = CodeReviewResult.model_json_schema()
    messages = [
        {
            "role": "system",
            "content": f"""You are an expert code reviewer.
Return JSON matching this schema exactly:
{json.dumps(schema_json, indent=2)}"""
        },
        {"role": "user", "content": f"Review this Python code:\n```python\n{code}\n```"}
    ]

    for attempt in range(max_retries + 1):
        response = client.chat.completions.create(
            model=SMART_MODEL,
            temperature=0.1,
            max_tokens=1500,
            response_format={"type": "json_object"},
            messages=messages
        )
        raw_text = response.choices[0].message.content
        try:
            raw = json.loads(raw_text)
            return CodeReviewResult(**raw)
        except (json.JSONDecodeError, ValidationError) as e:
            print(f"⚠️ Attempt {attempt + 1} failed validation: {e}")
            messages.append({"role": "assistant", "content": raw_text})
            messages.append({
                "role": "user",
                "content": f"Your JSON failed validation with this error:\n{e}\n"
                            f"Please return corrected JSON matching the schema exactly."
            })

    raise RuntimeError(f"Failed to produce valid output after {max_retries + 1} attempts")


review2 = review_code_with_retry(sample_code)
print(f"✅ Got a valid CodeReviewResult after retry logic: score={review2.overall_score}/10")


In [ ]:
# 🎯 EXERCISE 5: Design a Pydantic schema + extraction prompt for parsing
# meeting notes into: { "action_items": [{"owner": str, "task": str, "due_date": str|null}],
#                        "decisions": [str], "follow_up_meeting_needed": bool }
# Test it on the meeting_notes text below.

meeting_notes = """
We agreed Sarah will finish the API spec by Friday. John will set up the staging
environment, no firm deadline yet. We decided to use Postgres instead of MongoDB.
We'll need another sync next week to review progress.
"""

# TODO: define your Pydantic model(s) here
# class MeetingSummary(BaseModel):
#     ...

# TODO: write the extraction function and test it


---
## 📋 Part 6: Prompt Templates & Chaining

### 6.1 Theory: prompts as code, not strings

In a real codebase, prompts are never hardcoded f-strings scattered across your code — they are **templates**, versioned and tested like any other piece of business logic. A templating engine like **Jinja2** gives you:

- **Separation of concerns** — prompt text lives separately from the Python code that fills it in.
- **Conditional sections** — include examples, constraints, or context only when relevant (`{% if %}` blocks), keeping prompts as short as possible when extra sections aren't needed (shorter prompt = lower cost + often better focus).
- **Loops** — render lists of examples, tools, or constraints without manual string concatenation.
- **Reusability** — the same template can power many different agents/tasks just by changing the config object passed into it.

### 6.2 Theory: Prompt Chaining

**Prompt chaining** splits a complex task into a sequence of smaller, focused LLM calls, where the output of step *i* becomes part of the input to step *i+1*. This is the single-LLM-call analogue of the classic software engineering principle "do one thing well":

| Benefit | Why it matters |
|---------|-----------------|
| **Higher accuracy per step** | A focused sub-task (e.g., "just extract structure") is easier for the model to do correctly than "do everything at once" |
| **Debuggability** | When something goes wrong, you know exactly which step produced bad output |
| **Caching** | Intermediate results (e.g., parsed code structure) can be cached and reused across different downstream chains |
| **Heterogeneous models** | Cheap/fast models for simple steps (extraction), expensive/smart models only for the steps that truly need them |
| **Composability** | Chains can be combined into bigger pipelines — this is exactly the conceptual building block behind frameworks like LangChain/LangGraph |

The cost is **latency and total token usage** (multiple round trips instead of one), so chaining is a deliberate trade-off, not a default for every task.


In [ ]:
# ============================================================
# 6.1 JINJA2 TEMPLATES for production prompts
# In production, prompts are NEVER hardcoded as plain strings
# ============================================================

from jinja2 import Template

AGENT_TEMPLATE = Template("""
# ROLE
{{ role }}

# TASK
{{ task }}

# CONTEXT
{{ context }}

{% if constraints %}
# CONSTRAINTS
{% for constraint in constraints %}
- {{ constraint }}
{% endfor %}
{% endif %}

{% if examples %}
# EXAMPLES
{% for ex in examples %}
Input: {{ ex.input }}
Output: {{ ex.output }}
---
{% endfor %}
{% endif %}

# OUTPUT FORMAT
{{ output_format }}
""")


@dataclass
class PromptConfig:
    role: str
    task: str
    context: str
    output_format: str
    constraints: list = field(default_factory=list)
    examples: list = field(default_factory=list)


# Build a prompt from the template
config = PromptConfig(
    role="Senior Python Code Reviewer with security expertise",
    task="Review the provided Python code for bugs and security issues",
    context="Production FastAPI service, 10,000 requests/minute, Python 3.12",
    output_format='JSON: {"bugs": [...], "security_issues": [...], "score": 0-10}',
    constraints=[
        "Do not suggest a complete rewrite",
        "Max 5 issues",
        "Focus on security and correctness over style"
    ]
)

rendered_prompt = AGENT_TEMPLATE.render(**config.__dict__)
print("📝 Rendered System Prompt:")
print(rendered_prompt)
print(f"\n📊 Token count: {count_tokens(rendered_prompt)} tokens")


In [ ]:
# 🧪 PRACTICE: Same template, different config — note how cheap it is to
# spin up a brand-new "agent persona" once your prompt is templated.
config2 = PromptConfig(
    role="Senior Frontend Engineer specializing in accessibility (a11y)",
    task="Review the provided React component for accessibility issues",
    context="Public-facing e-commerce site, must meet WCAG 2.1 AA",
    output_format='JSON: {"a11y_issues": [...], "score": 0-10}',
    constraints=["Max 5 issues", "Cite the specific WCAG criterion for each issue"],
    examples=[{"input": "<img src='x.png'>", "output": "Missing alt text — WCAG 1.1.1"}]
)
print(AGENT_TEMPLATE.render(**config2.__dict__))


In [ ]:
# ============================================================
# 6.2 PROMPT CHAINING PATTERN
# Split a complex task into multiple steps, step i's output → step i+1's input
# ============================================================

def analyze_codebase_with_chaining(code: str) -> dict:
    """
    3-step prompt chain: Extract → Analyze → Recommend
    Why chain instead of one giant prompt?
    - Each step is specialized → higher accuracy
    - Easier to debug: you know exactly which step failed
    - Intermediate results can be cached
    - Models handle smaller, focused tasks better
    """
    print("🔗 PROMPT CHAIN - 3 STEPS")
    print("=" * 60)

    # STEP 1: Extract structure
    print("\n📍 STEP 1: Extracting code structure...")
    step1_response = client.chat.completions.create(
        model=DEFAULT_MODEL,
        max_tokens=800,
        response_format={"type": "json_object"},
        messages=[
            {
                "role": "system",
                "content": "You are a code parser. Extract the structure as JSON: {functions, classes, imports, dependencies}"
            },
            {"role": "user", "content": f"Parse this Python code:\n{code}"}
        ]
    )
    structure = step1_response.choices[0].message.content
    print(f"   Structure extracted: {len(structure)} chars")

    # STEP 2: Security analysis
    print("\n📍 STEP 2: Security analysis...")
    step2_response = client.chat.completions.create(
        model=SMART_MODEL,
        max_tokens=800,
        response_format={"type": "json_object"},
        messages=[
            {
                "role": "system",
                "content": "You are a security expert. Analyze the code structure for vulnerabilities. Return JSON: {vulnerabilities: [{type, severity, description}]}"
            },
            {"role": "user", "content": f"Code structure:\n{structure}\n\nOriginal code:\n{code}"}
        ]
    )
    analysis = step2_response.choices[0].message.content
    print(f"   Analysis complete: {len(analysis)} chars")

    # STEP 3: Recommendations
    print("\n📍 STEP 3: Generating recommendations...")
    step3_response = client.chat.completions.create(
        model=SMART_MODEL,
        max_tokens=1000,
        messages=[
            {
                "role": "system",
                "content": "You are a senior architect. Based on the security analysis, propose a concrete, prioritized action plan with code examples."
            },
            {
                "role": "user",
                "content": f"Security analysis:\n{analysis}\n\nOriginal code:\n{code}\n\nGenerate an action plan with fixed code."
            }
        ]
    )
    plan = step3_response.choices[0].message.content

    print("\n📊 FINAL RESULT:")
    print("-" * 40)
    print(plan)

    return {
        "structure": json.loads(structure),
        "analysis": json.loads(analysis),
        "action_plan": plan
    }


# Test
insecure_code = """
import os
import sqlite3

def execute_command(cmd: str):
    result = os.system(cmd)  # Direct OS command execution!
    return result

def search_user(username: str, db_path: str):
    conn = sqlite3.connect(db_path)
    # SQL Injection vulnerability
    rows = conn.execute(f"SELECT * FROM users WHERE name = '{username}'").fetchall()
    return rows
"""

chain_result = analyze_codebase_with_chaining(insecure_code)


In [ ]:
# 🎯 EXERCISE 6: Add a 4th step to the chain above: "Estimate the engineering
# effort (in hours) to fix every issue found in step 2", using the SMART_MODEL.
# Hint: reuse `analysis` as input and append the result to the returned dict.

def add_effort_estimation_step(analysis_json: dict) -> str:
    # TODO: implement
    pass


---
## 🔄 Part 7: ReAct Pattern (Reason + Act)

### 7.1 Theory: ReAct, the backbone of AI Agent frameworks

**ReAct** (Yao et al., 2022, *"ReAct: Synergizing Reasoning and Acting in Language Models"*) interleaves two things the model does turn by turn:

- **Reasoning** ("Thought") — the model thinks out loud about what it knows and what it still needs.
- **Acting** ("Action") — the model emits a structured request to use a tool (search, calculator, database query, code execution, ...).

The loop looks like this:

```
Thought  → the model reasons about what to do next
Action   → the model requests a tool call, e.g. search("gpt-4o pricing")
Observation → YOUR CODE executes the tool and returns the real result
Thought  → the model reasons again, now grounded in real data
Action   → ... (repeat until enough information is gathered)
Final Answer → the model commits to a final response
```

This is exactly the orchestration loop underneath every "agent" you have heard of — LangChain agents, AutoGPT, function-calling-based assistants, etc. Understanding ReAct from first principles (without a framework hiding it from you) is one of the most valuable things you can do as an AI engineer, because it demystifies what "an agent" actually *is*: **an LLM call inside a `while` loop, with a parser and a tool dispatcher around it.**

### 7.2 Why ReAct beats pure Chain-of-Thought for many tasks

Plain CoT can hallucinate facts mid-reasoning because the model has no way to "check" itself against reality — it just keeps generating tokens conditioned on its own (possibly wrong) earlier tokens. ReAct breaks that feedback loop by injecting **real, externally-verified observations** between reasoning steps, which:
- **Reduces hallucination** — the model's next "Thought" is conditioned on ground truth, not on its own unverified guess.
- **Enables multi-step tasks** that require information the model doesn't have (current prices, file contents, database rows, calculator-grade arithmetic).
- **Makes the process auditable** — you get a full, inspectable trace of what the agent "thought" and what tools it actually called, which is essential for debugging and for trust/safety review.

### 7.3 Anatomy of a ReAct implementation

To build ReAct from scratch (without a framework) you need exactly four pieces, which map 1:1 onto the code below:
1. **A system prompt** that teaches the model the `Thought/Action/Observation/Final Answer` protocol and lists the available tools.
2. **A set of real tool functions** your code can actually execute.
3. **A parser** that extracts the requested action and its arguments from the model's raw text output.
4. **A loop** that: calls the model → checks for "Final Answer" → otherwise parses+executes the requested action → appends the observation to the conversation → repeats.


In [ ]:
# ============================================================
# 7. ReAct PATTERN — the backbone of AI Agent frameworks
# Thought → Action → Observation → Thought → Action → ...
# ============================================================

# Define mock tools (in production: call real APIs)
def search_web(query: str) -> str:
    """Mock: simulate a web search."""
    mock_results = {
        "gpt-4o pricing": "GPT-4o: $2.50/1M input tokens, $10.00/1M output tokens (as of 2024)",
        "openai rate limits": "GPT-4o: 10,000 RPM, 30M TPM for Tier 5 accounts",
        "python asyncio": "asyncio is Python's built-in library for async I/O, ideal for concurrent LLM calls",
    }
    for key, val in mock_results.items():
        if key.lower() in query.lower():
            return val
    return f"Search results for '{query}': Found 847 results. Top result: [relevant info about {query}]"


def calculate(expression: str) -> str:
    """Evaluate a math expression safely."""
    try:
        # Safety: only allow math operations, no builtins
        allowed_names = {"__builtins__": {}}
        result = eval(expression, allowed_names)
        return str(result)
    except Exception as e:
        return f"Error: {e}"


def read_file(path: str) -> str:
    """Mock: read a file."""
    return f"Content of {path}: [Mock file content with 2500 tokens of Python code]"


# Tool registry
TOOLS = {
    "search": search_web,
    "calculate": calculate,
    "read_file": read_file
}


def parse_react_action(text: str) -> tuple[Optional[str], Optional[str]]:
    """Parse 'Action: tool_name(params)' from the ReAct output."""
    action_match = re.search(r'Action:\s*(\w+)\((.*)\)', text)
    if action_match:
        tool_name = action_match.group(1).strip()
        tool_input = action_match.group(2).strip().strip('"\'')
        return tool_name, tool_input
    return None, None


REACT_SYSTEM_PROMPT = """
Solve tasks using this EXACT loop:

Thought: [Your reasoning about what to do next]
Action: [tool_name]("[parameters]")
Observation: [Result returned by the tool]
... (repeat as needed)
Thought: [Final reasoning with all info gathered]
Final Answer: [Complete answer to the user's question]

Available tools:
- search(query: str): Search the web for information
- calculate(expression: str): Evaluate a math expression
- read_file(path: str): Read a file from disk

IMPORTANT: Always start with Thought, then Action OR Final Answer.
If you have enough info, go straight to Final Answer.
"""


def run_react_agent(user_query: str, max_steps: int = 6) -> str:
    """
    A minimal ReAct agent loop.
    In production: use LangGraph or LangChain agents, but the underlying
    mechanism is exactly this loop.
    """
    messages = [
        {"role": "system", "content": REACT_SYSTEM_PROMPT},
        {"role": "user", "content": user_query}
    ]

    print(f"🤖 REACT AGENT running...")
    print(f"Query: {user_query}")
    print("=" * 60)

    for step in range(max_steps):
        print(f"\n--- Step {step + 1} ---")

        response = client.chat.completions.create(
            model=SMART_MODEL,
            temperature=0.1,
            max_tokens=500,
            messages=messages
        )

        agent_output = response.choices[0].message.content
        print(agent_output)

        # Check if done
        if "Final Answer:" in agent_output:
            print("\n✅ Agent completed!")
            return agent_output.split("Final Answer:")[-1].strip()

        # Parse and execute the action
        tool_name, tool_input = parse_react_action(agent_output)
        if tool_name and tool_name in TOOLS:
            observation = TOOLS[tool_name](tool_input)
            print(f"\nObservation: {observation}")

            # Add to conversation history
            messages.append({"role": "assistant", "content": agent_output})
            messages.append({
                "role": "user",
                "content": f"Observation: {observation}\n\nContinue with the next Thought:"
            })
        else:
            print("⚠️ No valid action found, stopping.")
            break

    return "Max steps reached."


# Test the ReAct Agent
answer = run_react_agent(
    "What is the total monthly cost of running 500,000 GPT-4o API calls per day, "
    "assuming 1000 input tokens and 500 output tokens per call?"
)


In [ ]:
# 🧪 PRACTICE: Inspecting the full trace, not just the final answer
# In production agent systems, you almost always want to LOG the full
# Thought/Action/Observation trace, not just the final answer — for debugging,
# auditing, and computing per-request cost (number of LLM calls × tokens).

print("📋 FINAL ANSWER:")
print(answer)


### 7.4 Common pitfalls when building ReAct agents from scratch

| Pitfall | Symptom | Fix |
|---------|---------|-----|
| **Loose action parsing** | Agent's intent is clear but the regex doesn't match (extra whitespace, different quote style) | Make the parser more permissive, or — better — use native function/tool calling APIs instead of parsing free text (covered in Part 2) |
| **No max_steps guard** | Agent loops forever, racking up cost | Always cap `max_steps` and handle the "ran out of steps" case explicitly |
| **Tool errors crash the loop** | One bad tool call kills the whole agent run | Wrap every tool call in try/except and feed the error back as an Observation — let the model decide how to recover |
| **Unbounded conversation growth** | Each step appends messages; long-running agents blow the context window | Summarize or truncate older steps; only keep the most recent N observations in full |
| **The model "fakes" an observation** | Model writes both Action AND a plausible-looking Observation in the same turn, skipping the real tool call | Stop generation right after "Action:" (via stop sequences) so the model cannot hallucinate the observation itself |


In [ ]:
# 🎯 EXERCISE 7: Build Your Own ReAct Agent
# Add 2 new tools to the agent above and test it end-to-end.

def get_current_time(timezone: str = "UTC") -> str:
    """TODO: implement (can be a mock that returns a fixed string per timezone)."""
    pass


def get_weather(city: str) -> str:
    """TODO: implement (mock weather data is fine for this exercise)."""
    pass


# TODO: 1) register both tools in a new TOOLS dict
# TODO: 2) update REACT_SYSTEM_PROMPT's tool list to mention them
# TODO: 3) ask the agent a question that requires BOTH new tools, e.g.:
#          "What time is it in Tokyo, and what's the weather like there right now?"


---
## 📋 Part 1 Summary & What Comes Next

### ✅ What you've learned in this notebook

| Technique | Status |
|-----------|--------|
| LLM mental model, tokens, context window | ✅ |
| Sampling parameters (temperature, top_p, seed, etc.) | ✅ |
| Zero-shot / One-shot / Few-shot Prompting | ✅ |
| Dynamic Few-shot Selection (keyword + embeddings) | ✅ |
| Chain-of-Thought (Zero-shot & Few-shot CoT) | ✅ |
| Self-Consistency (Majority Voting) | ✅ |
| Tree-of-Thought | ✅ |
| Role Prompting & System Prompt Anatomy | ✅ |
| Multi-Persona prompting for multi-agent systems | ✅ |
| JSON Mode & Pydantic-validated Structured Output | ✅ |
| Validation-retry loop pattern | ✅ |
| Jinja2 Prompt Templates | ✅ |
| Prompt Chaining (multi-step pipelines) | ✅ |
| ReAct Pattern (Thought/Action/Observation loop) built from scratch | ✅ |

### 🔜 Coming up in Part 2 (separate notebook)

| # | Topic |
|---|-------|
| 8 | Advanced Techniques — Meta-Prompting, Self-Critique & Reflection |
| 9 | Prompt Evaluation & A/B Testing — building a real eval harness |
| 10 | Production — the "Smart Prompt Manager" (versioning, caching, cost tracking) |
| Bonus | Common Anti-Patterns to avoid |
| Bonus | A capstone set of hands-on exercises combining everything from Part 1 & 2 |

### 💡 Key mental models to carry forward

1. **An LLM is a function of everything in its context window.** Every technique you learned is a way of engineering that context more deliberately.
2. **Few-shot, CoT, and ReAct are all forms of "giving the model more useful conditioning before it commits to an answer."**
3. **Structured output + validation is what turns a chatbot into a programmable component.**
4. **An "agent" is just an LLM call wrapped in a loop with tools and a parser.** Frameworks (LangChain, LangGraph, CrewAI) automate this loop — but you should always be able to explain what they're doing under the hood, because you just built it yourself in Part 7.

See you in Part 2! 🚀


In [ ]:
# Quick reference card for Part 1
print("""
╔══════════════════════════════════════════════════════════════╗
║      QUICK REFERENCE — Prompt Engineering Cheatsheet (Part 1) ║
╠══════════════════════════════════════════════════════════════╣
║                                                                ║
║  ZERO-SHOT      → no examples, instruction only               ║
║  FEW-SHOT       → 3-5 high-quality, diverse, consistent examples ║
║  ZERO-SHOT CoT  → append 'let's think step by step'           ║
║  FEW-SHOT CoT   → show worked reasoning examples               ║
║  SELF-CONSISTENCY → sample N times, majority vote              ║
║  TREE-OF-THOUGHT → explore & score multiple reasoning branches ║
║  ROLE PROMPTING → specific identity + capabilities + constraints ║
║  JSON MODE      → response_format={"type":"json_object"}      ║
║  PYDANTIC       → validate everything before trusting it       ║
║  TEMPLATES      → Jinja2, never hardcode prompt strings        ║
║  CHAINING       → split complex tasks into focused LLM calls   ║
║  REACT          → Thought → Action → Observation loop          ║
║                                                                ║
╚══════════════════════════════════════════════════════════════╝
""")
